In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import regex
from tqdm.auto import tqdm

from src.processor import LogProcessor
from src.utils import (
    getFilesByDate,
    guardarExcel,
    isEmpty,
    isValidCode,
    loadElementosTopos,
    loadEstaciones,
    loadLocalizaciones,
    map_cata_mie,
    map_cod2name,
    map_mie_cata,
    parallelizeFunction,
    rellenarId,
    removeDoubleQuotes,
)

In [ ]:
# Cargar info mies
dir_info_mie = Path("data/MIEs.csv")
with dir_info_mie.open("r", encoding="utf8") as f:
    data = [
        row.split(";")
        for row in regex.sub(r"(?<=;) +", "", f.read()).split("\n")
        if row and not row.startswith("#")
    ]
df_info_mie = pd.DataFrame(data[1:], columns=data[0]).replace([""], [None])

In [ ]:
df_estaciones = loadEstaciones()
localizaciones = loadLocalizaciones()

In [ ]:
info_estaciones = pd.merge(
    df_estaciones,
    df_info_mie[["Catálogo", "MIE"]],
    on="Catálogo",
    how="outer",
)
info_estaciones = info_estaciones[
    np.invert(info_estaciones["Código"].apply(isEmpty))
    # & (
    #     ~info_estaciones["Mnemónico_comercial"].apply(
    #         lambda x: bool(regex.search(r"LAV\d", x))
    #     )
    # )
    # & (~info_estaciones["NombreCTC"].apply(lambda x: bool(regex.search(r"\bAV\b", x))))
].reset_index(drop=True)
info_estaciones["Código"] = info_estaciones["Código"].apply(rellenarId)

In [ ]:
info_estaciones = (
    info_estaciones.groupby("Código")
    .agg(
        {
            # "Nombre": lambda x: sorted(list(x), key=len)[0] if len(x) else None,
            "NombreCTC": lambda x: sorted(list(x), key=len)[0] if len(x) else None,
            "Delegación": lambda x: sorted(list(x), key=len)[0] if len(x) else None,
            # "Nombre": list,
            # "NombreCTC": list,
            # "Delegación": list,
            "CTC": list,
            "Catálogo": list,
            "Mnemónico": list,
            "Mnemónico_comercial": list,
            "MIE": list,
            "Tecnólogo": list,
        },
    )
    .reset_index()
    .explode(
        ["CTC", "Catálogo", "Mnemónico", "Mnemónico_comercial", "MIE", "Tecnólogo"]
    )
)

In [ ]:
info_estaciones = pd.merge(
    info_estaciones,
    localizaciones,
    how="inner",
    on="Código",
)

# info_estaciones = info_estaciones[info_estaciones["Latitud"] > 0]

### Carga de la info de los catálogos

#### Catálogos convencionales

In [ ]:
dir_catalogos = Path(r"data/catálogos")
objetos = []
# catalogos = ["CHA"]
for catalogo in dir_catalogos.glob(f"*.csv"):
    obj = pd.read_csv(
        catalogo,
        header=None,
        names=["Mnemónico", "Elemento", "Tipo"],
        sep=";",
        index_col=False,
        na_filter=False,
    )
    obj["Catálogo"] = regex.split(r"[\s-_]+", catalogo.name)[0]
    objetos.append(obj)
objetos = pd.concat(objetos)

filtered_cata = (
    objetos[["Catálogo", "Mnemónico", "Elemento", "Tipo"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
filtered_cata["Tipo"] = filtered_cata["Tipo"].astype(str)

filtered_cata = filtered_cata[
    np.invert(filtered_cata.map(isEmpty).any(axis=1))
].reset_index(drop=True)
filtered_cata["Mnemónico_comercial"] = filtered_cata["Mnemónico"]

In [ ]:
# dir_catalogos = Path(r"data/catálogos")
# objetos = []
# catalogos = info_estaciones["Catálogo"].unique()
# # catalogos = ["CHA"]
# for c in catalogos:
#     if not c:
#         continue
#     cats = list(dir_catalogos.glob(f"*{c}*.csv"))
#     if not cats:
#         print(f"No se encuentra el catálogo '{c}'")
#         continue
#     obj = pd.read_csv(
#         sorted(cats)[0],
#         header=None,
#         names=["Mnemónico", "Elemento", "Tipo"],
#         sep=";",
#         index_col=False,
#         na_filter=False,
#     )
#     obj["Catálogo"] = c
#     objetos.append(obj)
# objetos = pd.concat(objetos)

# filtered_cata = (
#     objetos[["Catálogo", "Mnemónico", "Elemento", "Tipo"]]
#     .drop_duplicates()
#     .reset_index(drop=True)
# )
# filtered_cata["Tipo"] = filtered_cata["Tipo"].astype(str)

# filtered_cata = filtered_cata[
#     np.invert(filtered_cata.map(isEmpty).any(axis=1))
# ].reset_index(drop=True)
# filtered_cata["Mnemónico_comercial"] = filtered_cata["Mnemónico"]

#### Catálogos AV

In [ ]:
# lavs = list(dir_catalogos.glob(f"*.xlsx"))
# for lav_dir in lavs:
#     print(lav_dir)
#     lav = pd.read_excel(lav_dir)
#     lav.columns = [c.strip() for c in lav.columns]
#     lav["Mnemónico_comercial"] = lav["IDENTIFICADOR"].apply(
#         lambda x: regex.sub(r"^CV\.", "", x).rsplit(".", 1)[0]
#     )
#     lav[["LAV", "Mnemónico", "Elemento"]] = (
#         lav["IDENTIFICADOR"]
#         .apply(lambda x: regex.sub(r"^CV\.", "", x).split("."))
#         .tolist()
#     )
#     lav["Catálogo"] = regex.sub(r"\s+(CV)$", "", lav_dir.stem).split(" ")[-1]
#     lav["Tipo"] = "0"
#     lav["Mnemónico_comercial"] = lav["LAV"] + "." + lav["Mnemónico"]
#     lav = lav[["Catálogo", "Mnemónico", "Mnemónico_comercial", "Elemento", "Tipo"]]
#     filtered_cata = pd.concat([filtered_cata, lav])

### Topos

In [ ]:
elementos_topos = loadElementosTopos()

### Carga de la info de la mensajería

In [ ]:
start_date = "2025-05-18"
end_date = "2025-05-20"
dir_mies = Path(r"C:\Users\jose.espinosa\Documents\Data\mie_mse")

w_logs = getFilesByDate(dir_mies, start_date, end_date)
if w_logs:
    fnames, full_days = list(zip(*w_logs))
    days = f"{full_days[0].strftime('%Y-%m-%d')} - {full_days[-1].strftime('%Y-%m-%d')}"
else:
    fnames = []
    days = ""
log_processor = LogProcessor()
df_logs = log_processor.loadFilesLogs(
    fnames, "mie_mse", load="elemento", days=days, format_fechas=False
)

In [ ]:
df_logs["MIE"] = df_logs["MIE"].str.lower()
mie_elements = (
    df_logs[["MIE", "Mnemónico", "Elemento", "Tipo"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

### Relación

In [ ]:
filtered_cata["MIE"] = filtered_cata["Catálogo"].apply(map_cata_mie.get)
# info_estaciones["NombreMIE"] = info_estaciones["MIE"].apply(map_mie_nombre.get)
filtered_cata["in_cata"] = True
mie_elements["in_mie"] = True
elementos_topos["in_topo"] = True

print("filtered_cata")
display(filtered_cata.head(2))
print("mie_elements")
display(mie_elements.head(2))
print("elementos_topos")
display(elementos_topos.head(2))
print("info_estaciones")
display(info_estaciones.head(2))

In [ ]:
df_topos_comercial = pd.merge(
    elementos_topos,
    info_estaciones.drop(columns=["Mnemónico"]),
    how="left",
    left_on=["Código", "Mnemónico_comercial"],
    right_on=["Código", "Mnemónico_comercial"],
    # suffixes=["", "_del"],
)
topologia = df_topos_comercial[
    [
        "Código",
        "Mnemónico_comercial",
        "Mnemónico",
        "Elemento",
        "Tipo",
        "in_topo",
        "MIE",
    ]
].drop_duplicates()

In [ ]:
df_total = pd.merge(
    filtered_cata,
    mie_elements,
    how="outer",
    on=["MIE", "Mnemónico", "Elemento", "Tipo"],
)
df_total = pd.merge(
    df_total,  # .drop(columns=["Catálogo"]),
    info_estaciones[
        [
            "Código",
            "Catálogo",
            # "MIE",
            "Mnemónico",
            "Mnemónico_comercial",
        ]
    ],  # enclavamiento
    how="right",
    # how="left",
    left_on=[
        "Catálogo",
        # "MIE",
        "Mnemónico",
        "Mnemónico_comercial",
    ],
    right_on=[
        "Catálogo",
        # "MIE",
        "Mnemónico",
        "Mnemónico_comercial",
    ],
)
df_total = pd.merge(
    df_total.drop(columns=["Mnemónico_comercial"]),
    topologia,
    how="outer",
    on=["MIE", "Mnemónico", "Elemento", "Tipo"],
    suffixes=["_enclavamiento", "_dependencia"],
)
df_total[["in_mie", "in_topo", "in_cata"]] = df_total[
    ["in_mie", "in_topo", "in_cata"]
].fillna(False)

In [ ]:
estado_enclavamiento = (
    df_total[
        [
            "MIE",
            "Mnemónico",
            "Código_enclavamiento",
            "Elemento",
            "Tipo",
            "Mnemónico_comercial",
            "Código_dependencia",
            "in_mie",
            "in_topo",
            "in_cata",
        ]
    ]
    .replace([pd.NA], [None])
    .groupby(
        ["MIE", "Mnemónico", "Código_enclavamiento", "Elemento", "Tipo"],
        dropna=False,
    )
    .agg(
        {
            "Mnemónico_comercial": list,
            "Código_dependencia": list,
            "in_mie": lambda x: list(x)[0],
            "in_cata": lambda x: list(x)[0],
            "in_topo": lambda x: list(x)[0],
            # "in_topo": list,
        }
    )
    .reset_index()
)
estado_enclavamiento["Comentarios"] = ""
estado_enclavamiento["Catálogo"] = estado_enclavamiento["MIE"].apply(map_mie_cata.get)

In [ ]:
estado_enclavamiento.loc[
    ~estado_enclavamiento[["Mnemónico", "Mnemónico_comercial"]].apply(
        lambda x: x["Mnemónico"] in x["Mnemónico_comercial"], axis=1
    ),
    "Comentarios",
] += "Falta elemento en topo principal\n"
estado_enclavamiento.loc[
    (estado_enclavamiento["in_topo"] & np.invert(estado_enclavamiento["in_mie"])),
    "Comentarios",
] += "No se recibe de la MIE\n"

estado_enclavamiento["Mnemónico_comercial"] = estado_enclavamiento[
    "Mnemónico_comercial"
].apply(lambda x: ",".join(el for el in list(x) if el))
estado_enclavamiento["Código_dependencia"] = estado_enclavamiento[
    "Código_dependencia"
].apply(lambda x: ",".join(el for el in list(x) if el))

In [ ]:
estado_enclavamiento = estado_enclavamiento.loc[
    # estado_enclavamiento["Tipo"].isin(["0", "1", "20", "102", "105", "107"]),
    :,
    [
        "MIE",
        "Catálogo",
        "Mnemónico",
        "Código_enclavamiento",
        "Mnemónico_comercial",
        "Código_dependencia",
        "Elemento",
        "Tipo",
        "in_mie",
        "in_cata",
        "in_topo",
        "Comentarios",
    ],
].sort_values(by=["Catálogo", "Mnemónico"])

In [ ]:
resumen = estado_enclavamiento[
    [
        "MIE",
        "Catálogo",
        "Mnemónico",
        "Código_enclavamiento",
        # "Elemento",
        # "Tipo",
        "in_mie",
        "in_topo",
        "in_cata",
    ]
].copy()
resumen["Completo"] = resumen["in_topo"] & resumen["in_cata"]
resumen = (
    resumen.groupby(
        by=[
            "MIE",
            "Catálogo",
            "Mnemónico",
            "Código_enclavamiento",
        ]
    )
    .agg("sum")
    .reset_index()
)
resumen["%Completo"] = resumen["Completo"] / resumen["in_cata"] * 100

In [ ]:
resumen = pd.merge(
    resumen.rename(columns={"Código_enclavamiento": "Código"}),
    info_estaciones,
    how="left",
    on=["Código", "MIE", "Catálogo", "Mnemónico"],
)[
    [
        "MIE",
        "Catálogo",
        "Mnemónico",
        "Código",
        "Nombre",
        "NombreCTC",
        "Delegación",
        "CTC",
        "Tecnólogo",
        # "NombreMIE",
        "in_mie",
        "in_topo",
        "in_cata",
        "Completo",
        "%Completo",
        "Longitud",
        "Latitud",
    ]
].sort_values(
    by=["Catálogo", "Mnemónico"]
)

In [ ]:
guardarExcel(
    resumen,
    f"Estado estaciones comerciales {start_date} - {end_date}.xlsx",
    sheet_name="Resumen",
    append_sheet=False,
)
guardarExcel(
    estado_enclavamiento,
    f"Estado estaciones comerciales {start_date} - {end_date}.xlsx",
    sheet_name="Todo",
    append_sheet=True,
)